# Neural network playground (Julia)

Small feedforward MLP **from scratch**: dense layers as matrix–vector maps, ReLU, linear output, MSE, and manual backprop.

**Kernel:** Julia (IJulia). From the repo root: `julia --project=. -e 'using Pkg; Pkg.instantiate()'` then install the kernel once: `julia --project=. -e 'using IJulia; IJulia.installkernel("JuliaNN", "--project=$(pwd())")'`.


In [2]:
using LinearAlgebra, Random, Printf

Random.seed!(42);


## `Dense` layer

`W` has shape `(n_out, n_in)` so `z = W * a .+ b` with column vector `a`.


In [4]:
struct Dense
    W::Matrix{Float64}
    b::Vector{Float64}
end

function Dense(rng::AbstractRNG, n_in::Int, n_out::Int)
    scale = sqrt(2.0 / n_in)
    Dense(scale .* randn(rng, n_out, n_in), zeros(n_out))
end

relu(x) = @. max(zero(x), x)
drelu(z) = Float64.(z .> 0)


drelu (generic function with 1 method)

## Forward (cached for backprop)

Last layer is **linear** (no ReLU on the readout).


In [ ]:
struct ActivCache
    a_ins::Vector{Vector{Float64}}
    zs::Vector{Vector{Float64}}
end

function forward!(layers::Vector{Dense}, x::AbstractVector{<:Real})
    a = Vector{Float64}(x)
    a_ins = Vector{Float64}[]
    zs = Vector{Float64}[]
    for i in eachindex(layers)
        push!(a_ins, copy(a))
        L = layers[i]
        z = L.W * a .+ L.b
        push!(zs, z)
        a = i < lastindex(layers) ? relu(z) : z
    end
    return a, ActivCache(a_ins, zs)
end


## MSE loss and backward pass


In [ ]:
mse_loss(yhat, y) = (diff = yhat .- y; (sum(abs2, diff) / 2, diff))

function backward!(layers::Vector{Dense}, cache::ActivCache, y, yhat)
    Ln = length(layers)
    _, diff = mse_loss(yhat, y)
    δ = Vector{Float64}(diff)
    dWs = [zero(l.W) for l in layers]
    dbs = [zero(l.b) for l in layers]
    for l in Ln:-1:1
        dWs[l] .+= δ * cache.a_ins[l]'
        dbs[l] .+= δ
        l == 1 && break
        δ = layers[l].W' * δ
        δ .*= drelu(cache.zs[l - 1])
    end
    return dWs, dbs
end


## SGD


In [ ]:
function sgd!(layers::Vector{Dense}, dWs, dbs, η::Float64)
    for i in eachindex(layers)
        layers[i].W .-= η .* dWs[i]
        layers[i].b .-= η .* dbs[i]
    end
    return layers
end

function zero_like_grads(layers)
    ([zero(l.W) for l in layers], [zero(l.b) for l in layers])
end

function train_step!(layers, batch, η)
    dWs, dbs = zero_like_grads(layers)
    n = length(batch)
    loss_acc = 0.0
    for (x, y) in batch
        yhat, cache = forward!(layers, x)
        li, _ = mse_loss(yhat, y)
        loss_acc += li
        gW, gb = backward!(layers, cache, y, yhat)
        for i in eachindex(layers)
            dWs[i] .+= gW[i]
            dbs[i] .+= gb[i]
        end
    end
    for i in eachindex(layers)
        dWs[i] ./= n
        dbs[i] ./= n
    end
    sgd!(layers, dWs, dbs, η)
    return loss_acc / n
end


## Conditioning / spectrum of `W`


In [ ]:
for (i, L) in enumerate(layers)
    @printf("Layer %d: size(W)=%s, cond(W)=%.2f, max|λ|≈%.3f\n",
            i, size(L.W), cond(L.W), maximum(abs, eigvals(L.W)))
end
